# Foundations 4 — `bedrock-runtime`: Converse, inference profiles, and the catalogue

Notebook 01 covered `bedrock-mantle`: bearer tokens, the three URL path families,
and the OpenAI- and Anthropic-shaped APIs. This one covers the other endpoint.

`bedrock-runtime` is the AWS-native surface. Three things make it different, and
all three trip people up:

1. **Auth is SigV4 through the AWS SDK.** No token to mint, no expiry to manage.
2. **Many models cannot be called by their model ID.** They require a
   cross-Region *inference profile* and reject the bare ID outright.
3. **The response is a list of typed blocks**, not a string. Indexing `[0]` and
   reading `text` works right up until the model returns a reasoning block first.

Read this before the family notebooks if you have not used Converse before.


In [1]:
import sys

sys.path.insert(0, "../_shared")

import json

from bedrock import (
    control_client,
    converse,
    converse_reasoning,
    converse_text,
    inference_profiles,
    resolve_runtime_id,
    runtime_client,
    runtime_models,
)

REGION = "us-east-1"
CLAUDE = "anthropic.claude-sonnet-5"  # inference-profile only
NOVA = "amazon.nova-micro-v1"  # on-demand capable

runtime = runtime_client(REGION)
print("client:", type(runtime).__name__)
print("no bearer token needed - boto3 signs each request with SigV4")


client: BedrockRuntime
no bearer token needed - boto3 signs each request with SigV4


## 1. Auth: nothing to mint

On `bedrock-mantle` you mint a short-term bearer token that expires within 12
hours. On `bedrock-runtime` the SDK signs each request with your ambient
credentials, so there is no token lifecycle to manage at all. That is the single
biggest operational difference between the two endpoints.


In [2]:
text, response = converse(
    NOVA,
    [{"role": "user", "content": [{"text": "Reply with exactly: OK"}]}],
    max_tokens=16,
    region=REGION,
)
print("answer     :", text.strip())
print("stop reason:", response.get("stopReason"))
print("usage      :", response.get("usage"))


answer     : OK
stop reason: end_turn
usage      : {'inputTokens': 5, 'outputTokens': 2, 'totalTokens': 7}


## 2. The request shape

Converse normalises the request across providers, which is its main selling
point: the same call works for Nova, Claude and Llama. Three things differ from
the OpenAI shape:

- `content` is a **list of blocks**, not a string
- the token budget lives in `inferenceConfig`, not at the top level
- the system prompt is its own `system` parameter, not a message with
  `role: "system"`


In [3]:
raw = runtime.converse(
    modelId=resolve_runtime_id(NOVA, REGION),
    system=[{"text": "You are terse."}],
    messages=[
        {"role": "user", "content": [{"text": "Name one benefit of queues."}]},
        # A prior assistant turn goes here too, same block shape - that is how
        # you carry multi-turn state. There is no server-side conversation store
        # on this endpoint; you resend the history each time.
    ],
    inferenceConfig={"maxTokens": 80, "temperature": 0.3},
)
print(json.dumps({k: v for k, v in raw.items() if k != "ResponseMetadata"},
                 indent=2, default=str)[:700])


{
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "Order preservation."
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 10,
    "outputTokens": 4,
    "totalTokens": 14
  },
  "metrics": {
    "latencyMs": 352
  }
}


## 3. Never index `content[0]`

The response `content` list can contain `text`, `reasoningContent`, `toolUse` and
other block types, **in whatever order the model produced them**. A reasoning
model puts its trace first, so `content[0]["text"]` raises `KeyError`.

The cell below proves it with `moonshot.kimi-k2-thinking`, which reliably returns
a `reasoningContent` block ahead of its answer. It is not only reasoning models,
though, and it is not stable per model: Claude Sonnet 5 returned a non-text first
block on one call and a text first block on the next while this notebook was
being written. So do not special-case the models you know about — walk the list
and select by key every time. `converse_text()` and `converse_reasoning()` in
`_shared/bedrock.py` do exactly that.


In [4]:
REASONER = "moonshot.kimi-k2-thinking"

text, response = converse(
    REASONER,
    [{"role": "user", "content": [{"text": "What is 17 * 23? Think it through."}]}],
    max_tokens=600,
    region=REGION,
)

blocks = response.get("output", {}).get("message", {}).get("content", [])
print(f"{REASONER} returned {len(blocks)} content block(s):")
for i, block in enumerate(blocks):
    print(f"  content[{i}] -> {next(iter(block))}")

print("\nthe naive read:")
try:
    print("  content[0]['text'] =", blocks[0]["text"][:40])
except KeyError:
    print("  content[0]['text'] -> KeyError, because block 0 is the reasoning trace")

print("\nthe safe read:")
print("  converse_text()     :", converse_text(response).strip()[:70])
reasoning = converse_reasoning(response)
print("  converse_reasoning():", f"{len(reasoning)} chars" if reasoning else "(none)")

# Same request against Claude. Block ordering here varies between calls, which is
# the reason to never rely on it.
text2, response2 = converse(
    CLAUDE,
    [{"role": "user", "content": [{"text": "What is 17 * 23? Think it through."}]}],
    max_tokens=600,
    region=REGION,
)
kinds = [
    next(iter(b))
    for b in response2.get("output", {}).get("message", {}).get("content", [])
]
print(f"\n{CLAUDE} returned blocks: {kinds}")
print("  -> may or may not lead with text; treat the order as undefined.")


moonshot.kimi-k2-thinking returned 1 content block(s):
  content[0] -> reasoningContent

the naive read:
  content[0]['text'] -> KeyError, because block 0 is the reasoning trace

the safe read:
  converse_text()     : 
  converse_reasoning(): 1796 chars



anthropic.claude-sonnet-5 returned blocks: ['text']
  -> may or may not lead with text; treat the order as undefined.


## 4. Inference profiles, and why a model ID can be rejected

Bedrock addresses a model on `bedrock-runtime` two ways:

    bare model ID          amazon.nova-micro-v1:0
    inference profile ID   us.amazon.nova-micro-v1:0

A profile routes your request across several Regions in one geography, which
raises availability and effective throughput. This is Cross-Region Inference
(CRIS).

The catch: models listed as `INFERENCE_PROFILE` only — which today includes
almost the whole Claude family — **reject the bare ID**. Models listed as
`ON_DEMAND` accept either form.


In [5]:
profile = f"us.{CLAUDE}"
detail = control_client(REGION).get_inference_profile(
    inferenceProfileIdentifier=profile
)
print(f"{detail['inferenceProfileId']}  ({detail.get('type')}, {detail.get('status')})")
print("routes your request to:")
for model in detail.get("models", []):
    arn = model["modelArn"]
    print(f"    {arn.split(':')[3]:<12} {arn.split('/')[-1]}")

print(f"\ntotal profiles in {REGION}: {len(inference_profiles(REGION))}")

print("\nwhat the bare ID does:")
try:
    runtime.converse(
        modelId=CLAUDE,
        messages=[{"role": "user", "content": [{"text": "Reply OK"}]}],
        inferenceConfig={"maxTokens": 16},
    )
    print("    accepted")
except Exception as exc:
    print(f"    {type(exc).__name__}: {str(exc)[-120:]}")

print("\nresolve_runtime_id() picks the right form for you:")
for model in (CLAUDE, NOVA):
    print(f"    {model:<28} -> {resolve_runtime_id(model, REGION)}")


us.anthropic.claude-sonnet-5  (SYSTEM_DEFINED, ACTIVE)
routes your request to:
    us-east-1    anthropic.claude-sonnet-5
    us-east-2    anthropic.claude-sonnet-5
    us-west-2    anthropic.claude-sonnet-5

total profiles in us-east-1: 63

what the bare ID does:


    ValidationException: mand throughput isn’t supported. Retry your request with the ID or ARN of an inference profile that contains this model.

resolve_runtime_id() picks the right form for you:
    anthropic.claude-sonnet-5    -> us.anthropic.claude-sonnet-5
    amazon.nova-micro-v1         -> us.amazon.nova-micro-v1:0


## 5. Reading the catalogue

`ListFoundationModels` is how you answer "what can I call, and how" without
guessing. Two fields matter most:

- `inputModalities` / `outputModalities` — what the model consumes and produces
- `inferenceTypesSupported` — `ON_DEMAND`, `INFERENCE_PROFILE`, or `PROVISIONED`

A model with only `INFERENCE_PROFILE` needs the `us.` form from section 4. A
model with only `PROVISIONED` needs a purchased throughput commitment and will
refuse on-demand calls entirely.


In [6]:
catalogue = runtime_models(REGION)
print(f"{len(catalogue)} catalogue entries in {REGION}\n")

import collections

by_type = collections.Counter()
for entry in catalogue.values():
    if "ON_DEMAND" in entry["infer"]:
        by_type["ON_DEMAND (bare ID works)"] += 1
    elif "INFERENCE_PROFILE" in entry["infer"]:
        by_type["INFERENCE_PROFILE only (us. required)"] += 1
    else:
        by_type["PROVISIONED only"] += 1
for label, count in by_type.most_common():
    print(f"  {count:>4}  {label}")

print("\nmodels that return text and accept images, on-demand:")
shown = 0
for key, entry in sorted(catalogue.items()):
    if entry["out"] == {"TEXT"} and "IMAGE" in entry["in"] and "ON_DEMAND" in entry["infer"]:
        print(f"    {key}")
        shown += 1
        if shown == 6:
            print("    ...")
            break


100 catalogue entries in us-east-1

    63  ON_DEMAND (bare ID works)
    37  INFERENCE_PROFILE only (us. required)

models that return text and accept images, on-demand:
    amazon.nova-lite-v1
    amazon.nova-pro-v1
    anthropic.claude-3-haiku-20240307-v1
    google.gemma-3-12b-it
    google.gemma-3-27b-it
    google.gemma-3-4b-it
    ...


## 6. Streaming, and the lower-level escape hatch

`converse_stream` gives you incremental output with the same normalised shape.
`invoke_model` is the raw passthrough: you send the provider's own JSON body and
get theirs back. Reach for `invoke_model` only when a provider exposes something
Converse has not normalised yet — otherwise Converse is less code and portable.


In [7]:
stream = runtime.converse_stream(
    modelId=resolve_runtime_id(NOVA, REGION),
    messages=[{"role": "user", "content": [{"text": "Count to five."}]}],
    inferenceConfig={"maxTokens": 60},
)
events = collections.Counter()
pieces = []
try:
    for event in stream["stream"]:
        kind = next(iter(event))
        events[kind] += 1
        if kind == "contentBlockDelta":
            pieces.append(event[kind]["delta"].get("text", ""))
except Exception as exc:
    # A stream can fail AFTER delivering part of the answer: a mid-stream
    # 5xx is not rare, and it has happened while building these notebooks.
    # Report what arrived instead of losing it - production code has to
    # decide whether a partial answer is usable or the call must be retried.
    print(f"\n[stream interrupted: {type(exc).__name__}]")
print("event types:", dict(events))
print("assembled  :", "".join(pieces).strip()[:90])

# invoke_model: the provider's own schema, not Converse's.
body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 24,
    "messages": [{"role": "user", "content": "Reply with exactly: OK"}],
}
raw = runtime.invoke_model(
    modelId=resolve_runtime_id(CLAUDE, REGION), body=json.dumps(body)
)
payload = json.loads(raw["body"].read())
print("\ninvoke_model top-level keys:", sorted(payload.keys()))
# Same discipline as section 3: select the text block, do not index [0].
answer = "".join(b["text"] for b in payload["content"] if b.get("type") == "text")
print("text:", answer.strip()[:60])


event types: {'messageStart': 1, 'contentBlockDelta': 12, 'contentBlockStop': 1, 'messageStop': 1, 'metadata': 1}
assembled  : Sure, here we go:

1, 2, 3, 4, 5.

If you need anything else or have a different request, 



invoke_model top-level keys: ['content', 'id', 'model', 'role', 'stop_details', 'stop_reason', 'stop_sequence', 'type', 'usage']
text: OK


## 7. Choosing an endpoint

Ask in this order:

1. **Is the model on `bedrock-runtime` at all?** Gemma 4, Grok and GPT-5.x are
   `bedrock-mantle` only. That decides it.
2. **Is it on `bedrock-mantle` at all?** Nova, Llama, Cohere and the Palmyra text
   models are `bedrock-runtime` only. That also decides it.
3. **Both?** Then choose on the code you already have. Existing OpenAI- or
   Anthropic-shaped code moves to `bedrock-mantle` with a base-URL change.
   Anything else is usually less work on Converse, and you avoid the token
   lifecycle.

`endpoints_for(model_id)` answers 1 and 2 from the live catalogues. Each family
notebook states the answer for its own models and shows the working.

## Takeaways

- `bedrock-runtime` needs no token minting; SigV4 does it.
- `content` is an ordered list of typed blocks. Select by key, never index `[0]` —
  the same model can put a reasoning block first on one call and not the next.
- `INFERENCE_PROFILE`-only models reject the bare ID. Learn the error text; most
  of the Claude family produces it.
- Read `inferenceTypesSupported` before writing the call, not after debugging it.
- Prefer Converse over `invoke_model` unless you need something unnormalised.
